# Реакции пользователя

In [126]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit # для разделения с учетом групп
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from solution_utils import(read_dialogs, read_labels, get_groups, clean_normalize_dialog, calculate_metrics, SEED)

In [127]:
DATA_PATH = "hw1_train.jsonl"
LABELS = np.array(["positive", "negative", "neutral"])

dialogs = read_dialogs(DATA_PATH)
print("диалогов:", len(dialogs))
reactions = read_labels(DATA_PATH, "reaction")
labeled_dialogs = [d for d in dialogs if reactions.get(d.get('dialog_id')) is not None]
print("размеченных диалогов", len(labeled_dialogs))

диалогов: 10000
размеченных диалогов 3000


In [128]:
user_texts = [clean_normalize_dialog(d) for d in labeled_dialogs]
n_gr, groups_ids = get_groups(user_texts)
groups_ids = np.array(groups_ids)
print(f"групп: {n_gr}")
sizes = np.bincount(groups_ids)
print(f"групп >1: {(sizes > 1).sum()}")
print(f"макс размер: {sizes.max()}")
print(f"диалогов в группах >1: {sizes[sizes > 1].sum()}")

групп: 2978
групп >1: 17
макс размер: 4
диалогов в группах >1: 39


Разделяем на train, val и test с учетом баланса классов и группировки

In [129]:
X = np.array(user_texts)
y = np.array([reactions[d["dialog_id"]] for d in labeled_dialogs])

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
idx_train_val, idx_test = next(gss.split(X, y, groups_ids))
gss = GroupShuffleSplit(n_splits=1, train_size=0.75, random_state=SEED)
idx_train, idx_val = next(gss.split(X[idx_train_val], y[idx_train_val], groups_ids[idx_train_val]))

X_train, y_train = X[idx_train_val[idx_train]], y[idx_train_val[idx_train]] # 60%
X_val, y_val = X[idx_train_val[idx_val]], y[idx_train_val[idx_val]] # 20%
X_test, y_test = X[idx_test], y[idx_test] # 20%

In [130]:
for name, labels_part in [
    ("train", y_train),
    ("validation", y_val),
    ("test", y_test),
]:
    print(f"\n{name}: {len(labels_part)} примеров")
    print(
        pd.Series(labels_part)
        .value_counts(normalize=True)
        .reindex(LABELS, fill_value=0)
        .round(3)
    )


train: 1804 примеров
positive    0.099
negative    0.318
neutral     0.583
Name: proportion, dtype: float64

validation: 599 примеров
positive    0.114
negative    0.302
neutral     0.584
Name: proportion, dtype: float64

test: 597 примеров
positive    0.094
negative    0.281
neutral     0.625
Name: proportion, dtype: float64


Получаем Tf-idf для всех данных

In [131]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # добавляем биграммы
    sublinear_tf=True,     # уменьшает скорость роста влияния слова от его частоты
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf   = vectorizer.transform(X_val)
X_test_tfidf  = vectorizer.transform(X_test)

## Базовые модели

### Наивный Байес

Мы на семинаре использовали MultinomialNB, но сейчас больше подойдет ComplementNB, так как он специально спроектирован для несбалансированных данных, как у нас

In [132]:
alphas = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 0.9, 1.0, 1.5]
metrics_NB = []
for alpha in alphas:
    nb = ComplementNB(alpha=alpha)
    nb.fit(X_train_tfidf, y_train)
    y_pred = nb.predict(X_val_tfidf)
    metrics_NB.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_NB_df = pd.DataFrame(metrics_NB, index=alphas)
metrics_NB_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.001,0.621035,0.437770,0.428598,0.588169,"[[2, 18, 48], [7, 85, 89], [10, 55, 285]]","{'precision_positive': 0.10526315789473684, 'r..."
0.005,0.627713,0.449086,0.441967,0.597119,"[[3, 19, 46], [7, 89, 85], [9, 57, 284]]","{'precision_positive': 0.15789473684210525, 'r..."
0.010,0.631052,0.450991,0.443557,0.598748,"[[3, 19, 46], [6, 89, 86], [7, 57, 286]]","{'precision_positive': 0.1875, 'recall_positiv..."
0.020,0.634391,0.447168,0.436982,0.597214,"[[2, 17, 49], [4, 87, 90], [6, 53, 291]]","{'precision_positive': 0.16666666666666666, 'r..."
0.050,0.637730,0.438009,0.425232,0.591940,"[[1, 12, 55], [3, 79, 99], [4, 44, 302]]","{'precision_positive': 0.125, 'recall_positive..."
0.100,0.647746,0.431771,0.413350,0.588834,"[[0, 8, 60], [1, 70, 110], [0, 32, 318]]","{'precision_positive': 0.0, 'recall_positive':..."
0.200,0.647746,0.424657,0.405263,0.581773,"[[0, 5, 63], [0, 62, 119], [0, 24, 326]]","{'precision_positive': 0.0, 'recall_positive':..."
0.500,0.647746,0.418432,0.397505,0.575089,"[[0, 2, 66], [0, 55, 126], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."
0.900,0.619366,0.387124,0.355322,0.532757,"[[0, 2, 66], [0, 38, 143], [0, 17, 333]]","{'precision_positive': 0.0, 'recall_positive':..."
1.000,0.622705,0.389029,0.357158,0.535213,"[[0, 2, 66], [0, 38, 143], [0, 15, 335]]","{'precision_positive': 0.0, 'recall_positive':..."


По всем параметрам лучше всего alpha = 0.01, выбираем его

In [133]:
alpha_opt = 0.01
nb = ComplementNB(alpha=alpha_opt).fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_val_tfidf)

## Логистическая регрессия

In [134]:
C_grid = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 1.5, 2.0, 5.0]

metrics_logreg = []
for C in C_grid:
    log_reg = LogisticRegression(
        class_weight="balanced", # веса классов обратно пропорционально частоте
        max_iter=5000,
        C=C,
    )
    log_reg.fit(X_train_tfidf, y_train)
    y_pred = log_reg.predict(X_val_tfidf)
    metrics_logreg.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_logreg_df = pd.DataFrame(metrics_logreg, index=C_grid)
metrics_logreg_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.001,0.639399,0.561606,0.565564,0.638219,"[[28, 14, 26], [14, 97, 70], [20, 72, 258]]","{'precision_positive': 0.45161290322580644, 'r..."
0.010,0.637730,0.556704,0.561162,0.636231,"[[27, 14, 27], [14, 97, 70], [20, 72, 258]]","{'precision_positive': 0.4426229508196721, 're..."
0.050,0.642738,0.558672,0.565173,0.640056,"[[27, 14, 27], [13, 96, 72], [19, 69, 262]]","{'precision_positive': 0.4576271186440678, 're..."
0.100,0.646077,0.560577,0.570273,0.642485,"[[27, 14, 27], [11, 96, 74], [17, 69, 264]]","{'precision_positive': 0.4909090909090909, 're..."
0.500,0.651085,0.536180,0.554246,0.640109,"[[21, 13, 34], [7, 92, 82], [13, 60, 277]]","{'precision_positive': 0.5121951219512195, 're..."
1.000,0.656093,0.524128,0.546620,0.640213,"[[17, 14, 37], [3, 93, 85], [8, 59, 283]]","{'precision_positive': 0.6071428571428571, 're..."
1.500,0.656093,0.523239,0.545943,0.639787,"[[17, 14, 37], [3, 92, 86], [8, 58, 284]]","{'precision_positive': 0.6071428571428571, 're..."
2.000,0.659432,0.522083,0.545283,0.641739,"[[16, 14, 38], [2, 93, 86], [7, 57, 286]]","{'precision_positive': 0.64, 'recall_positive'..."
5.000,0.656093,0.511783,0.535763,0.635311,"[[15, 14, 39], [2, 88, 91], [5, 55, 290]]","{'precision_positive': 0.6818181818181818, 're..."


Выбираем C = 0.1

In [135]:
C_opt_logreg = 0.1
log_reg = LogisticRegression(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_logreg,
).fit(X_train_tfidf, y_train)
y_pred_logreg = log_reg.predict(X_val_tfidf)

## Метод опорных векторов

In [136]:
metrics_svc = []
for C in C_grid:
    svc = LinearSVC(
        class_weight="balanced", # веса классов обратно пропорционально частоте
        max_iter=5000,
        C=C,
    )
    svc.fit(X_train_tfidf, y_train)
    y_pred = svc.predict(X_val_tfidf)
    metrics_svc.append(calculate_metrics(y_val, y_pred, LABELS))

metrics_svc_df = pd.DataFrame(metrics_svc, index=C_grid)
metrics_svc_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
0.001,0.584307,0.333333,0.245873,0.430996,"[[0, 0, 68], [0, 0, 181], [0, 0, 350]]","{'precision_positive': 0.0, 'recall_positive':..."
0.010,0.584307,0.333333,0.245873,0.430996,"[[0, 0, 68], [0, 0, 181], [0, 0, 350]]","{'precision_positive': 0.0, 'recall_positive':..."
0.050,0.662771,0.438067,0.423935,0.597326,"[[1, 8, 59], [0, 63, 118], [0, 17, 333]]","{'precision_positive': 1.0, 'recall_positive':..."
0.100,0.659432,0.461742,0.465887,0.612450,"[[5, 9, 54], [0, 74, 107], [1, 33, 316]]","{'precision_positive': 0.8333333333333334, 're..."
0.500,0.664441,0.499360,0.521174,0.634686,"[[12, 12, 44], [1, 82, 98], [4, 42, 304]]","{'precision_positive': 0.7058823529411765, 're..."
1.000,0.669449,0.504885,0.525771,0.640835,"[[12, 13, 43], [2, 85, 94], [4, 42, 304]]","{'precision_positive': 0.6666666666666666, 're..."
1.500,0.669449,0.500936,0.519768,0.639377,"[[11, 13, 44], [2, 85, 94], [4, 41, 305]]","{'precision_positive': 0.6470588235294118, 're..."
2.000,0.671119,0.501888,0.520609,0.640851,"[[11, 14, 43], [2, 85, 94], [4, 40, 306]]","{'precision_positive': 0.6470588235294118, 're..."
5.000,0.666110,0.498142,0.516622,0.636087,"[[11, 14, 43], [2, 84, 95], [4, 42, 304]]","{'precision_positive': 0.6470588235294118, 're..."


Лучший С = 1

In [137]:
C_opt_svc = 1
svc = LinearSVC(
    class_weight="balanced", # веса классов обратно пропорционально частоте
    max_iter=5000,
    C=C_opt_svc,
).fit(X_train_tfidf, y_train)
y_pred_svc = svc.predict(X_val_tfidf)

In [138]:
metrics_models = []
metrics_models.append(calculate_metrics(y_val, y_pred_nb, LABELS))
metrics_models.append(calculate_metrics(y_val, y_pred_logreg, LABELS))
metrics_models.append(calculate_metrics(y_val, y_pred_svc, LABELS))

metrics_models_df = pd.DataFrame(metrics_models, index=["NB", "LR", "SVC"])
metrics_models_df

,accuracy,balanced_accuracy,macro-f1,weighted-f1,confusion_matrix,per_class_metrics
NB,0.631052,0.450991,0.443557,0.598748,"[[3, 19, 46], [6, 89, 86], [7, 57, 286]]","{'precision_positive': 0.1875, 'recall_positiv..."
LR,0.646077,0.560577,0.570273,0.642485,"[[27, 14, 27], [11, 96, 74], [17, 69, 264]]","{'precision_positive': 0.4909090909090909, 're..."
SVC,0.669449,0.504885,0.525771,0.640835,"[[12, 13, 43], [2, 85, 94], [4, 42, 304]]","{'precision_positive': 0.6666666666666666, 're..."


Лучше всего себя показывает логистическая регрессия, берем ее за основу нашей модели и на ней проверяем гипотезы.

## Формируем "свою" модель на основе проверки гипотез о данных и параметрах

Изучим, как влияет выбор только первых/последних реплик на предсказания. Предположения: основная информация о реакции пользователя содержится в конце.

In [152]:
texts_by_variant = {
    "all": user_texts,
    "first": [
        clean_normalize_dialog(d, num=1, position='first')
        for d in labeled_dialogs
    ],
    "last": [
        clean_normalize_dialog(d, num=1, position='last')
        for d in labeled_dialogs
    ],
    "last_two": [
        clean_normalize_dialog(d, num=2, position='last')
        for d in labeled_dialogs
    ],
    "last_three": [
            clean_normalize_dialog(d, num=3, position='last')
            for d in labeled_dialogs
        ],
}

In [153]:
variant_results = []
variant_models = {}

for variant, texts in texts_by_variant.items():
    X_train_text = [texts[i] for i in idx_train_val[idx_train]]
    X_val_text   = [texts[i] for i in idx_train_val[idx_val]]

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
    )
    X_train_vec = vectorizer.fit_transform(X_train_text)
    X_val_vec   = vectorizer.transform(X_val_text)

    model = LogisticRegression(
        C=0.1,
        class_weight="balanced",
        max_iter=5000,
        random_state=SEED,
    )
    model.fit(X_train_vec, y_train)
    pred = model.predict(X_val_vec)

    m = calculate_metrics(y_val, pred, LABELS)

    variant_results.append({
        "variant":         variant,
        "macro-f1":        m["macro-f1"],
        "weighted-f1":     m["weighted-f1"],
        "accuracy":        m["accuracy"],
        "balanced_acc":    m["balanced_accuracy"],
        "F1_positive":     m["per_class_metrics"]["f1_positive"],
        "F1_neutral":      m["per_class_metrics"]["f1_neutral"],
        "F1_negative":     m["per_class_metrics"]["f1_negative"],
    })

    variant_models[variant] = {
        "vectorizer": vectorizer,
        "model": model,
    }

variant_results_df = pd.DataFrame(variant_results).round(4).sort_values("macro-f1", ascending=False)

variant_results_df

,variant,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
3,last_two,0.5865,0.6544,0.6644,0.5627,0.4727,0.7513,0.5353
4,last_three,0.5836,0.6439,0.6477,0.5716,0.4878,0.7333,0.5296
0,all,0.5703,0.6425,0.6461,0.5606,0.4390,0.7385,0.5333
1,first,0.4989,0.5933,0.5977,0.4939,0.2927,0.6957,0.5083
2,last,0.4847,0.5766,0.5943,0.4650,0.3158,0.6972,0.4412


Ожидаемо, first — слабый, но, на удивление, last с одной репликой еще хуже. А вот 2 или 3 последние реплики уже лучше всего текста по большинству параметров. last_three выигрывает по balanced_accuracy и F1_positive, но остальные параметры лучше у last_two, пока берем его, но держим в голове и другой вариант.

Но нам этого недостаточно, плюс если мы совсем не будем учитывать остальной диалог, можем потерять длинные споры и недовольства в середине или (и так очень редкую) положительную реакцию. Попробуем просто добавить последним репликам веса.

Попробовали просто в hstack присоединить tfidf по последним строкам, умноженное на разные веса, но это почти ничего не дало. Не будем приводить здесь этот эксперимент, давайте попробуем добавить вес последним строкам иначе: отдельно обучаем 2 модели, одну на полных данных, вторую на последних репликах, и итоговые вероятности определяем как W * p_last + (1 - W) * p_full, p_last - вероятность, предсказанная по последним репликам, p_full - по всем

In [154]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=3)
X_tr_last2_tfidf = vectorizer.fit_transform([texts_by_variant.get("last_two")[i] for i in idx_train_val[idx_train]])
X_va_last2_tfidf = vectorizer.transform([texts_by_variant.get("last_two")[i] for i in idx_train_val[idx_val]])
log_reg  = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000).fit(X_train_tfidf, y_train)
model_last2 = LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000).fit(X_tr_last2_tfidf, y_train)

results = []
weights = [0.0, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 1.0]
for W in weights:
    p_full  = log_reg.predict_proba(X_val_tfidf)
    p_last2 = model_last2.predict_proba(X_va_last2_tfidf)
    p_res = W * p_last2 + (1 - W) * p_full # взвешенное среднее вероятностей
    pred = log_reg.classes_[p_res.argmax(axis=1)]
    m = calculate_metrics(y_val, pred, LABELS)
    results.append({
            "macro-f1":        m["macro-f1"],
            "weighted-f1":     m["weighted-f1"],
            "accuracy":        m["accuracy"],
            "balanced_acc":    m["balanced_accuracy"],
            "F1_positive":     m["per_class_metrics"]["f1_positive"],
            "F1_neutral":      m["per_class_metrics"]["f1_neutral"],
            "F1_negative":     m["per_class_metrics"]["f1_negative"],
        })

results = pd.DataFrame(results, index=weights).round(4).sort_values("macro-f1", ascending=False)
results

,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
0.60,0.6053,0.6674,0.6745,0.5847,0.5043,0.7582,0.5533
0.65,0.6033,0.6660,0.6728,0.5837,0.5000,0.7565,0.5533
0.55,0.6029,0.6652,0.6728,0.5819,0.5043,0.7578,0.5465
0.40,0.6006,0.6654,0.6728,0.5798,0.4912,0.7571,0.5533
0.70,0.5995,0.6626,0.6694,0.5809,0.4957,0.7537,0.5491
0.50,0.5993,0.6645,0.6728,0.5780,0.4912,0.7584,0.5481
0.30,0.5905,0.6549,0.6628,0.5705,0.4870,0.7497,0.5349
0.20,0.5886,0.6515,0.6578,0.5716,0.4915,0.7469,0.5272
1.00,0.5835,0.6408,0.6461,0.5702,0.5041,0.7335,0.5130
0.00,0.5703,0.6425,0.6461,0.5606,0.4390,0.7385,0.5333


По результатам валидации учший вес W = 0.6 для 2 последних строк. Причем вес 0 для последних строк - худший в данном эксперименте, и прирост идет по всем рассмотренным параметрам (кроме F1_negative, оно такое же), особенно для нашего самого редкого позитивного класса! 

#### ??? Разные уровни очистки ???

Мы взяли самый общий уровень очистки данных - только реплики пользователя, нормализованные, без стоп-слов и очищенные от кода. Может быть, что-то из этого было лишним?

In [155]:
import re
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from solution_utils import delete_code_lines, normalize

_stem = SnowballStemmer("english")
_WORD = re.compile(r"[^\W\d_]+")

def normalize_no_stop_words(text: str) -> list[str]:
    """БЕЗ удаления стоп-слов."""
    words = _WORD.findall(text.lower())
    return [_stem.stem(word) for word in words if len(word) >= 2]

def processing_dialog(dialog, include_assistant, clean_code, drop_stopwords: bool):
    """почти clean_normalize_dialog но с опциональным удалением стоп-слов"""
    if include_assistant:
        text = [turn["text"] for turn in dialog["turns"]]
    else:
        text = [turn["text"] for turn in dialog["turns"] if turn["role"] == "user"]
    
    text = "\n".join(text)
    if clean_code:
        text = delete_code_lines(text)
    if drop_stopwords:
        clean_norm_text = " ".join(normalize(text))
    else:
        clean_norm_text = " ".join(normalize_no_stop_words(text))
    return clean_norm_text


In [156]:
variants = [
    ("basic", False, True, True),
    ("no clean_code", False, False, True),
    ("no stopwords", False, True,  False),
    ("with assistant", True, True, True),
    ("with assistant, no clean", True, False, True),
]

rows = []
for name, inc_a, clean, drop in variants:
    texts = [processing_dialog(d, inc_a, clean, drop) for d in labeled_dialogs]

    vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
    X_tr = vec.fit_transform([texts[i] for i in idx_train_val[idx_train]])
    X_va = vec.transform([texts[i] for i in idx_train_val[idx_val]])

    l_r = LogisticRegression(C=0.1, class_weight="balanced",
                             max_iter=5000, random_state=SEED).fit(X_tr, y_train)
    pred = l_r.predict(X_va)

    m = calculate_metrics(y_val, pred, LABELS)
    rows.append({
        "variant":       name,
        "macro-f1":      m["macro-f1"],
        "weighted-f1":   m["weighted-f1"],
        "accuracy":      m["accuracy"],
        "balanced_acc":  m["balanced_accuracy"],
        "F1_positive":   m["per_class_metrics"]["f1_positive"],
        "F1_neutral":    m["per_class_metrics"]["f1_neutral"],
        "F1_negative":   m["per_class_metrics"]["f1_negative"],
    })

variants_df = pd.DataFrame(rows).round(4).sort_values("macro-f1", ascending=False)
variants_df

,variant,macro-f1,weighted-f1,accuracy,balanced_acc,F1_positive,F1_neutral,F1_negative
1,no clean_code,0.5777,0.6534,0.6578,0.5681,0.4355,0.7510,0.5465
2,no stopwords,0.5731,0.6515,0.6477,0.5874,0.4224,0.7504,0.5465
0,basic,0.5703,0.6425,0.6461,0.5606,0.4390,0.7385,0.5333
3,with assistant,0.5698,0.6610,0.6711,0.5530,0.3654,0.7562,0.5879
4,"with assistant, no clean",0.5665,0.6574,0.6678,0.5494,0.3654,0.7541,0.5801


                           variant  macro-f1  weighted-f1  accuracy  balanced_acc  F1_positive  F1_neutral  F1_negative
                     no clean_code    0.6272       0.6807    0.6861        0.6141       0.5556      0.7688       0.5572
base (clean_code, stopwords, user)    0.6218       0.6747    0.6795        0.6103       0.5512      0.7620       0.5523
                    with assistant    0.5980       0.6740    0.6761        0.5916       0.4262      0.7525       0.6154
     with assistant, no clean_code    0.5968       0.6741    0.6761        0.5907       0.4228      0.7543       0.6133
       no clean_code, no stopwords    0.5867       0.6638    0.6628        0.5947       0.4314      0.7564       0.5723
              no stopwords removal    0.5812       0.6568    0.6561        0.5891       0.4342      0.7507       0.5588
 with assistant, no clean, no stop    0.5728       0.6539    0.6561        0.5682       0.3906      0.7383       0.5895
      with assistant, no stopwords    0.

In [ ]:
#TODO добавить в качестве признака количество важных знаков препинания !!! ???
# проверить влияние данных вроде длины реплик, попробовать разные уровни очистки текста и нормализации
# добавить в работу с кодом выкидывание по ''' !!!!!!